```mermaid
graph LR
    A[CineBot Agent] -->|MCPAdapter| B[(MCP Server)]
    B --> C[check_showtimes]
    B --> D[cancel_booking]
    B --> E[get_seat_map]
```

In [2]:
import warnings,os
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
assert os.environ.get("OPENAI_API_KEY") is not None, "OPENAI_API_KEY is not set in the environment variables."

In [4]:
import langchain.mcp

# MCP Adapter in LangChain

In [9]:
%%writefile cinebot_mcp_server.py
from fastmcp import FastMCP
from fastmcp.server.auth import AccessToken, TokenVerifier
from mcp.types import ToolAnnotations

AUTH_TOKEN = "cinebot-secret-token"


class StaticTokenVerifier(TokenVerifier):
    """Accepts requests whose bearer token exactly matches AUTH_TOKEN."""

    async def verify_token(self, token: str) -> AccessToken | None:
        if token == AUTH_TOKEN:
            return AccessToken(token=token, client_id="cinebot-client", scopes=[])
        return None


mcp = FastMCP("CineBot", auth=StaticTokenVerifier())

@mcp.tool()
def check_showtimes(movie_title: str) -> str:
    """Check available showtimes for a movie."""
    fake_showtimes = {
        "interstellar": "7:00 PM and 10:15 PM",
        "dune part two": "9:30 PM",
    }
    return fake_showtimes.get(movie_title.lower(), "No showtimes found.")

@mcp.tool(annotations=ToolAnnotations(destructiveHint=True,requiredAuthentication=True))
def cancel_booking(booking_id: str) -> str:
    """Cancel an existing booking. Irreversible."""
    return f"Booking {booking_id} cancelled."

@mcp.tool()
def get_seat_map(movie_title: str) -> dict:
    """Get the seat map for a movie -- returns structured data, not just text."""
    return {"movie": movie_title, "available_rows": ["A", "B", "C"], "sold_out_rows": ["D"]}

if __name__ == "__main__":
    mcp.run()
    # mcp.run(transport="http", port=8000)


Overwriting cinebot_mcp_server.py


In [5]:
from pathlib import Path
from langchain.mcp import MCPAdapter
from fastmcp.client import Client
from fastmcp.client.transports import PythonStdioTransport

In [6]:
cinebot_mcp_transport = PythonStdioTransport(
    script_path=Path("cinebot_mcp_server.py"),
    log_file=Path("cinebot_mcp_server.log")
)


async with MCPAdapter(cinebot_mcp_transport) as mcp_adapter:
    # Now you can use the mcp_adapter to interact with the MCP server
    tools=await mcp_adapter.list_tools()
    print("Available tools:", [t.name for t in tools])

Available tools: ['check_showtimes', 'cancel_booking', 'get_seat_map']


# Call a tool without any AI or Agent

In [7]:
from langchain.agents import create_agent

In [ ]:
# Directly invoking the tool without an agent

async with MCPAdapter(cinebot_mcp_transport) as adapter:
    tools = await adapter.list_tools()

    showtime_tool = next((t for t in tools if t.name == "check_showtimes"), None)

    direct_result = await showtime_tool.ainvoke({"movie_title": "interstellar"})
    print("Direct invocation result:", direct_result)

Direct invocation result: [{'type': 'text', 'text': '7:00 PM and 10:15 PM', 'id': 'lc_9e914d2b-2507-40f0-806b-15a7bc18c0b5'}]


In [ ]:
# Calling with Agent

try:
    cinebot_agent = create_agent(model='openai:gpt-5-mini',tools=tools)
    result = await cinebot_agent.ainvoke({"messages":[('user', "What are the showtimes for Interstellar?")]})
    print("Agent invocation result:", result)
except Exception as e:
    print("Error during agent invocation:", e)

Agent invocation result: {'messages': [HumanMessage(content='What are the showtimes for Interstellar?', additional_kwargs={}, response_metadata={}, id='e2d7da46-9209-4e08-bd80-91b679594c98'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 194, 'total_tokens': 220, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EVGaNzilwROG6qJcTIjlzdOAPEgUY', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1071b-e0f5-7dd0-abc1-41a0fc054a56-0', tool_calls=[{'name': 'check_showtimes', 'args': {'movie_title': 'Interstell

In [ ]:
from rich import print as rprint
rprint("[bold green]Agent tool invocation result:[/bold green]", result)

In [8]:
async def run_agent():
    # 1. Open adapter briefly just to discover tools
    async with MCPAdapter(cinebot_mcp_transport) as adapter:
        tools = await adapter.list_tools()
    
    # 2. Context exits here (connection closes)
    agent = create_agent("gpt-5-mini", tools)

    # 3. Agent runs:
    # - Call Tool A: Opens connection -> Runs Tool A -> Closes connection
    # - AI thinks...
    # - Call Tool B: Opens connection -> Runs Tool B -> Closes connection
    result = await agent.ainvoke({"messages": ["What is the weather in Tokyo?"]})
    return result

In [9]:
await run_agent()

{'messages': [HumanMessage(content='What is the weather in Tokyo?', additional_kwargs={}, response_metadata={}, id='6e98ee8d-52f2-4d1a-8448-b8e60af5e233'),
  AIMessage(content='I can’t fetch live weather data from here. If you want the current conditions in Tokyo right now, check a weather service (e.g., Weather.com, AccuWeather, the Weather Channel app, or OpenWeatherMap) or ask a smart assistant with internet access.\n\nIf it helps, here’s a brief summary of Tokyo’s typical climate so you know what to expect:\n\n- Winter (Dec–Feb): cool and relatively dry. Typical daytime 5–10°C (41–50°F); nights around 0–5°C (32–41°F). Snow in central Tokyo is uncommon but possible.\n- Spring (Mar–May): mild, pleasant. Roughly 8–20°C (46–68°F). Cherry-blossom season is late March–early April.\n- Summer (Jun–Aug): hot and humid. Daytime 25–35°C (77–95°F). Rainy season (tsuyu) usually in June–early July; typhoon risk in Aug–Sep.\n- Autumn (Sep–Nov): comfortable and clear. Daytime 15–25°C (59–77°F), co

# 5. Tool Output: Multimodal Content Blocks

An MCP tool result isn't limited to text. `langchain.mcp` converts whatever the server sends —
text, images, embedded files — into standard LangChain **content blocks**, so a model sees a
uniform shape regardless of what kind of server produced it.

| MCP content type | Converts to |
|---|---|
| `TextContent` | text content block |
| `ImageContent` | image content block (base64 + mime type) |
| `ResourceLink` (image mime type) | image content block (by URL) |
| `ResourceLink` (other) | file content block |
| `EmbeddedResource` (text) | text content block |
| `EmbeddedResource` (blob) | image or file content block, by mime type |
| `AudioContent` | **not yet supported** — raises `NotImplementedError` |

In [15]:
import asyncio
from langchain.agents import create_agent
from langchain.mcp import MCPAdapter
from langchain.messages import ToolMessage
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


async def access_multimodal_tool_content(server) -> dict:
    async with MCPAdapter(server) as adapter:
        tools = await adapter.list_tools()
        
        # Initialize an agent equipped with MCP tools
        agent = create_agent("gpt-5-mini", tools)
        
        # Invoke the agent with a prompt that triggers a multimodal tool
        result = await agent.ainvoke(
            {"messages": [{"role": "user", "content": "Take a screenshot."}]}
        )

        # Inspect returning content blocks
        for message in result["messages"]:
            if not isinstance(message, ToolMessage):
                continue
            
            for block in message.content_blocks:
                if block["type"] == "text":
                    print(f"Text: {block['text']}")
                elif block["type"] == "image":
                    preview = block.get("base64", "")[:20]
                    print(f"Image mime type: {block.get('mime_type')}")
                    print(f"Image base64 preview: {preview}...")

        return result


async def main():
    # Configure your MCP server connection parameters (e.g., using stdio)
    server_params = StdioServerParameters(
        command="npx",
        args=["-y", "@modelcontextprotocol/server-everything"],
    )

    # Establish stdio client session
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            
            # Execute the function passing the active session
            await access_multimodal_tool_content(session)


if __name__ == "__main__":
    asyncio.run(main())

UnsupportedOperation: fileno

In [13]:
await access_multimodal_tool_content(cinebot_mcp_transport)

{'messages': [HumanMessage(content='Take a screenshot.', additional_kwargs={}, response_metadata={}, id='ace73992-2d47-4fa7-8c1a-54ad7e5ec7ac'),
  AIMessage(content="I can’t take a screenshot of your device myself, but I can walk you through it. Which device or operating system are you using, and what exactly do you want to capture (whole screen, a window, a selected area, or a web page)?  \n\nQuick shortcuts and options by platform:\n\n- Windows\n  - Whole screen: Press Print Screen (copies to clipboard) or Win + PrtScn (saves to Pictures\\Screenshots).\n  - Active window: Alt + PrtScn (copies).\n  - Select area: Win + Shift + S (opens Snip & Sketch — choose rectangle, freeform, window, or full; image goes to clipboard and a notification to save).\n\n- macOS\n  - Whole screen: Cmd + Shift + 3 (saves to desktop).\n  - Selected area: Cmd + Shift + 4, then drag.\n  - Window: Cmd + Shift + 4, then Space and click a window.\n  - Copy to clipboard: add Ctrl (e.g., Cmd + Ctrl + Shift + 4).\n

# Structured Output 

In [20]:
async with MCPAdapter(cinebot_mcp_transport) as mcp_adapter:
    tools = await mcp_adapter.list_tools()
    seat_map_tool = next(t for t in tools if t.name == "get_seat_map")

    tool_call ={
        'name': 'get_seat_map',
        'args': {'movie_title': 'Interstellar'},
        'id': 'unique_call_id_12345',  # Optional: Provide a unique ID for the call
        'type':'tool_call'
    }

    message = await seat_map_tool.ainvoke(tool_call)
    print("Text Content (what my model reads)",message.content)
    print("Structured Data (what my code can use if required)",message.artifact)
    print(message.artifact['structured_content']['available_rows'])  # Accessing structured data directly

Text Content (what my model reads) [{'type': 'text', 'text': '{"movie":"Interstellar","available_rows":["A","B","C"],"sold_out_rows":["D"]}', 'id': 'lc_20624f81-6065-4d39-8767-c390a5c517a4'}]
Structured Data (what my code can use if required) {'structured_content': {'movie': 'Interstellar', 'available_rows': ['A', 'B', 'C'], 'sold_out_rows': ['D']}}
['A', 'B', 'C']


# Error handling



| Failure | What happened | What the agent sees |
|---|---|---|
| Server tool **ran and reported failure** (`isError=True`) | The tool executed, then explicitly failed (bad input, business-rule violation) | A `ToolMessage` with `status="error"` — **the agent can read it and self-correct**, same as any other tool error in this course |
| **Transport / session failure** | The connection dropped, the server crashed, the process couldn't start | **Raises an exception** — there is no error `ToolMessage` for the model to reason about, because nothing came back to convert |


In [21]:
%%writefile cinebot_error_server.py
from fastmcp import FastMCP
mcp = FastMCP("CineBot")

@mcp.tool()
def risky_lookup(booking_id: str) -> str:
    """Look up a booking -- fails if the ID format is wrong."""
    if not booking_id.startswith("BK"):
        raise ValueError(f"Invalid booking ID format: {booking_id!r}. Expected it to start with 'BK'.")
    return f"Booking {booking_id}: confirmed."

if __name__ == "__main__":
    mcp.run()


Overwriting cinebot_error_server.py


In [24]:
error_transport = PythonStdioTransport(
    script_path=Path('cinebot_error_server.py'),
    log_file=Path('cinebot_error_server.log')
)

async with MCPAdapter(error_transport) as mcp_adapter:
    tools = await mcp_adapter.list_tools()

risky_lookup_tool = tools[0]

bad_call = {'name': 'risky_lookup', 'args': {'booking_id': '12345'}, 'id': 'bad_call_001', 'type': 'tool_call'}

error_message = await risky_lookup_tool.ainvoke(bad_call)
print("Error message from tool:", error_message)
print("Content:", error_message.content)

Error message from tool: content=[{'type': 'text', 'text': "Error calling tool 'risky_lookup': Invalid booking ID format: '12345'. Expected it to start with 'BK'.", 'id': 'lc_b448052d-1f51-4b6c-85b7-859d817fd106'}] name='risky_lookup' tool_call_id='bad_call_001' status='error'
Content: [{'type': 'text', 'text': "Error calling tool 'risky_lookup': Invalid booking ID format: '12345'. Expected it to start with 'BK'.", 'id': 'lc_b448052d-1f51-4b6c-85b7-859d817fd106'}]


In [25]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

In [ ]:
def is_destructive_tool_call(tool):
    # return message.type == "tool_call" and message.name == "cancel_booking"
    meta = (tool.metadata or {}).get('mcp',{}).get('tool',{}).get('annotations',{})
    return bool(meta.get('destructive_hint'))


async with MCPAdapter(cinebot_mcp_transport) as mcp_adapter:
    tools = await mcp_adapter.list_tools()
    for t in tools:
        print(f"Tool: {t.name}, Destructive: {is_destructive_tool_call(t)}, Tools:{t}")


Tool: check_showtimes, Destructive: False, Tools:name='check_showtimes' description='Check available showtimes for a movie.' args_schema={'type': 'object', 'additionalProperties': False, 'properties': {'movie_title': {'type': 'string'}}, 'required': ['movie_title']} metadata={'mcp': {'tool': {'_meta': {'fastmcp': {'tags': []}}}, 'server': {'name': 'CineBot', 'version': '4.0.10'}}} handle_tool_error=<function _handle_mcp_tool_error at 0x1163c74c0> response_format='content_and_artifact' coroutine=<function as_langchain_tool.<locals>.call_tool at 0x12287fec0>
Tool: cancel_booking, Destructive: True, Tools:name='cancel_booking' description='Cancel an existing booking. Irreversible.' args_schema={'type': 'object', 'additionalProperties': False, 'properties': {'booking_id': {'type': 'string'}}, 'required': ['booking_id']} metadata={'mcp': {'tool': {'annotations': {'destructive_hint': True}, '_meta': {'fastmcp': {'tags': []}}}, 'server': {'name': 'CineBot', 'version': '4.0.10'}}} handle_tool_

In [ ]:
Tool: cancel_booking, Destructive: False, Tools:name='cancel_booking' description='Cancel an existing booking. Irreversible.' args_schema={'type': 'object', 'additionalProperties': False, 'properties': {'booking_id': {'type': 'string'}}, 'required': ['booking_id']} metadata={'mcp': {'tool': {'annotations': {'destructive_hint': True}, '_meta': {'fastmcp': {'tags': []}}}, 'server': {'name': 'CineBot', 'version': '4.0.10'}}} handle_tool_error=<function _handle_mcp_tool_error at 0x1163c74c0> response_format='content_and_artifact' coroutine=<function as_langchain_tool.<locals>.call_tool at 0x12287c7c0>


In [ ]:
async with MCPAdapter(cinebot_transport) as adapter:
    tools = await adapter.list_tools()

    for t in tools:
        print(f"{t.name}: destructive={is_destructive_tool_call(t)}")

    # One HITL config, built dynamically from whatever the server marked destructive.
    interrupt_on = {
        t.name: {"allowed_decisions": ["approve", "edit", "reject"]}
        for t in tools
        if is_destructive_tool_call(t)
    }
    print("\ninterrupt_on config:", interrupt_on)

    hitl_middleware = HumanInTheLoopMiddleware(interrupt_on=interrupt_on)
    cinebot_guarded = create_agent(model="openai:gpt-5-mini", tools=tools, middleware=[hitl_middleware])

check_showtimes: destructive=False
cancel_booking: destructive=True
get_seat_map: destructive=False

interrupt_on config: {'cancel_booking': {'allowed_decisions': ['approve', 'edit', 'reject']}}


---
# 9. Elicitation: When the Server Needs to Ask a Question Mid-Call

Some MCP tools need clarification partway through a call — "confirm you want to proceed?".
`langchain.mcp` answers this **automatically with a LangGraph `interrupt()`** — every client
`MCPAdapter` builds is armed for it by default, no extra code needed to enable it.

```mermaid
sequenceDiagram
    participant Agent
    participant MCPAdapter
    participant Server
    Agent->>MCPAdapter: call cancel_booking(...)
    MCPAdapter->>Server: tools/call
    Server-->>MCPAdapter: needs more input (elicitation)
    MCPAdapter->>Agent: interrupt() -- pauses the graph
    Note over Agent: A human answers, outside the graph
    Agent->>MCPAdapter: Command(resume={"responses": {key: answer}})
    MCPAdapter->>Server: tools/call, retried with the answer
    Server-->>MCPAdapter: final result
```


In [ ]:
%%writefile cinebot_elicit_server.py
from fastmcp import FastMCP, Context

mcp = FastMCP("CineBotElicit")

@mcp.tool()
async def cancel_booking_confirm(booking_id: str, ctx: Context) -> str:
    """Cancel a booking, but ask the caller to confirm first."""
    result = await ctx.elicit(
        f"Confirm cancellation of booking {booking_id}?",
        response_type=bool,
    )
    if result.action != "accept":
        return f"Cancellation of {booking_id} aborted ({result.action})."
    if not result.data:
        return f"Cancellation of {booking_id} aborted (user said no)."
    return f"Booking {booking_id}: cancelled."

if __name__ == "__main__":
    mcp.run()


Overwriting cinebot_elicit_server.py


In [ ]:
elicit_transport = PythonStdioTransport(
    script_path=Path("cinebot_elicit_server.py"),
    log_file=Path("cinebot_elicit_server.log"),
)

async with MCPAdapter(elicit_transport) as adapter:
    tools = await adapter.list_tools()
    confirm_tool = next(t for t in tools if t.name == "cancel_booking_confirm")

    tool_call = {
        "name": "cancel_booking_confirm",
        "args": {"booking_id": "BK123"},
        "id": "call_1",
        "type": "tool_call",
    }
    elicit_result = await confirm_tool.ainvoke(tool_call)
    print("status:", elicit_result.status)
    print("content:", elicit_result.content)


status: error
content: [{'type': 'text', 'text': 'elicitation via server-initiated requests is unavailable on 2026-07-28 connections.', 'id': 'lc_d6af8a0d-1a1e-4ed0-aea6-92a8edee0fd4'}]


# Connecting to Multiple Servers

# 10. Connecting to Multiple Servers

Real deployments rarely talk to just one MCP server. Two patterns, with a real tradeoff:

| Pattern | What it does | Tool naming | Protocol negotiation |
|---|---|---|---|
| **`MCPConfig` dict** | One aggregate connection across several servers | Prefixed by the config key you choose | **Shared** — the whole fleet negotiates down to the oldest protocol era any member requires |
| **`ClientGroup`** | Independent connections, one per server | Namespaced `{server}_{tool}` automatically | **Independent** — each member keeps its own protocol era and auth |

```mermaid
graph TB
    subgraph MCPConfig["MCPConfig -- one aggregate connection"]
        A1[Agent] --> C1[MCPAdapter]
        C1 --> S1[Server A]
        C1 --> S2[Server B]
        S1 -.shared era.-> S2
    end
    subgraph ClientGroup["ClientGroup -- independent connections"]
        A2[Agent] --> C2[MCPAdapter]
        C2 --> G[ClientGroup]
        G --> S3[Server A own era/auth]
        G --> S4[Server B own era/auth]
    end
```

In [ ]:
from fastmcp.mcp_config import MCPConfig

fleet_config: MCPConfig = {
    "mcpServers": {
        "cinebot": {
            "command": "python",
            "args": ["cinebot_mcp_server.py"],
        },
        "timetracker": {
            "url": "https://time-track-mcp-server.vercel.app/mcp/"
        }
    }
}

In [12]:

async with MCPAdapter(fleet_config) as adapter:
    fleet_tools = await adapter.list_tools()
    print("MCPConfig fleet tools:", [t.name for t in fleet_tools])

[10/05/26 11:02:36] Failed to connect to MCP server 'cinebot', skipping                                            
                    ╭───────────────────────────── Traceback (most recent call last) ─────────────────────────────╮
                    │ c:\Users\spatibandla\Shyam\Learnings\Krish\Agentic AI & AgentOps\Live-Class-2026\Weekend 14 │
                    │ - Oct 3 & Oct 4\MCP in Langchain\.venv\Lib\site-packages\exceptiongroup\_catch.py:63 in     │
                    │ handle_exception                                                                            │
                    │                                                                                             │
                    │    60 │   │   │   if matched:                                                               │
                    │    61 │   │   │   │   try:                                                                  │
                    │    62 │   │   │   │   │   try:                                                              │
                    │ ❱  63 │   │   │   │   │   │   raise matched                                                 │
                    │    64 │   │   │   │   │   except BaseExceptionGroup:                                        │
                    │    65 │   │   │   │   │   │   result = handler(matched)                                     │
                    │    66 │   │   │   │   except BaseExceptionGroup as new_exc:                                 │
                    ╰─────────────────────────────────────────────────────────────────────────────────────────────╯
                    ExceptionGroup:  (1 sub-exception)                                                             
                                                                                                                   
                    ╭───────────────────────────────────── Sub-exception #1 ──────────────────────────────────────╮
                    │ ╭─────────────────────────── Traceback (most recent call last) ───────────────────────────╮ │
                    │ │ c:\Users\spatibandla\Shyam\Learnings\Krish\Agentic AI &                                 │ │
                    │ │ AgentOps\Live-Class-2026\Weekend 14 - Oct 3 & Oct 4\MCP in                              │ │
                    │ │ Langchain\.venv\Lib\site-packages\fastmcp\client\client.py:1154 in _session_runner      │ │
                    │ │                                                                                         │ │
                    │ │ C:\Users\spatibandla\AppData\Roaming\uv\python\cpython-3.13.9-windows-x86_64-none\Lib\c │ │
                    │ │ ontextlib.py:668 in enter_async_context                                                 │ │
                    │ │                                                                                         │ │
                    │ │   665 │   │   │   raise TypeError(f"'{cls.__module__}.{cls.__qualname__}' object does " │ │
                    │ │   666 │   │   │   │   │   │   │   f"not support the asynchronous context manager protoc │ │
                    │ │   667 │   │   │   │   │   │      ) from None                                            │ │
                    │ │ ❱ 668 │   │   result = await _enter(cm)                                                 │ │
                    │ │   669 │   │   self._push_async_cm_exit(cm, _exit)                                       │ │
                    │ │   670 │   │   return result                                                             │ │
                    │ │   671                                                                                   │ │
                    │ │                                                                                         │ │
                    │ │                                ... 21 frames hidden ...                                 │ │
                    │ │                                 

[10/05/26 11:02:37] INFO     Proxy detected connected client - reusing existing session for all       proxy.py:1383
                             requests. This may cause context mixing in concurrent scenarios, and the              
                             session's existing settings apply, so backend results are validated                   
                             against their declared output schema rather than relayed as-is. Pass a                
                             disconnected client to avoid both.                                                    

MCPConfig fleet tools: ['timetracker_log_time', 'timetracker_get_timesheet', 'timetracker_get_project_summary', 'timetracker_list_projects']


In [ ]:
from fastmcp.client import Client
from fastmcp.client.group import ClientGroup


In [ ]:
group = ClientGroup(
    {
        "cinebot": Client(Path("cinebot_mcp_server.py")),
        "errors": Client(Path("cinebot_error_server.py"),mode="legacy"),
        'time_track_server':Client('https://time-track-mcp-server.vercel.app/mcp/')
    }
)
async with MCPAdapter(group) as adapter:
    group_tools = await adapter.list_tools()
    print("ClientGroup fleet tools:", [t.name for t in group_tools])


ClientGroup fleet tools: ['cinebot_check_showtimes', 'cinebot_cancel_booking', 'cinebot_get_seat_map', 'errors_risky_lookup', 'time_track_server_log_time', 'time_track_server_get_timesheet', 'time_track_server_get_project_summary', 'time_track_server_list_projects']


# Authentication

# 11. Authentication

MCP servers that aren't your own local script usually need auth. This is configured on the
**FastMCP client**, not on `MCPAdapter` itself — `MCPAdapter` just wraps whatever client you hand
it (or builds a default one for a bare URL).

| Need | Pattern |
|---|---|
| Bearer token | `Client(url, auth=token)` |
| OAuth 2.1 (discovery, browser redirect, token exchange) | `Client(url, auth="oauth")` |
| Persisted OAuth across runs | `Client(url, auth=OAuth(mcp_url=url, token_storage=...))` |
| Different auth per server | `ClientGroup` with a different `auth=` on each member `Client` |
| Per-user auth in a deployment | A custom auth handler resolves the caller server-side; the graph factory mints/exchanges a token per user |

```mermaid
sequenceDiagram
    participant App
    participant Client as fastmcp.Client
    participant Server
    App->>Client: Client(url, auth=token_or_"oauth")
    alt Bearer token
        Client->>Server: Authorization: Bearer <token>
        Server-->>Client: 200 OK
    else OAuth 2.1
        Client->>Server: discovery + dynamic client registration
        Server-->>Client: browser redirect
        Client->>Server: token exchange
        Server-->>Client: access token
    end
```

In [ ]:
async def load_tools_with_bearer(url: str, token: str):
    # `auth` accepts a bearer-token string, the literal "oauth", or any httpx.Auth.
    async with MCPAdapter(Client(url, auth=token)) as adapter:
        return await adapter.list_tools()

In [ ]:
# Must match AUTH_TOKEN in cinebot_mcp_server.py
CINEBOT_AUTH_TOKEN = "cinebot-secret-token"

# cinebot_mcp_server.py now serves over HTTP (port 8000) since the bearer
# token check only applies at a network boundary. Start it first with:
#   python cinebot_mcp_server.py
cinebot_tools = await load_tools_with_bearer("http://127.0.0.1:8000/mcp", CINEBOT_AUTH_TOKEN)
print("Tools over HTTP with bearer auth:", [t.name for t in cinebot_tools])

# 12. `langchain.mcp` vs. `langchain-mcp-adapters` — What Actually Changed

If you (or a client) already has an MCP integration built on the older, separate
`langchain-mcp-adapters` package, here's the honest diff — including what the new API **doesn't**
do yet.

| | `langchain-mcp-adapters` (older, separate package) | `langchain.mcp` (this notebook, beta, built-in) |
|---|---|---|
| Install | `pip install langchain-mcp-adapters` | `pip install "langchain[mcp]>=1.4.0"` — no separate package |
| Entry point | `MultiServerMCPClient({...}).get_tools()` | `async with MCPAdapter(target) as adapter: await adapter.list_tools()` |
| Resources (`get_resources`) | ✅ Supported | ❌ **Not exposed by `MCPAdapter`** — drop to the underlying `fastmcp.Client` directly if you need this |
| Prompts (`get_prompt`) | ✅ Supported | ❌ **Not exposed by `MCPAdapter`** — same workaround |
| Tool call interceptors (`tool_interceptors`, logging/retry/`Command`-update patterns) | ✅ Supported, via `MCPToolCallRequest` | ❌ **No interceptor hook in this beta** — customize by wrapping tools yourself after `list_tools()` |
| Progress callbacks | ✅ Supported | Not part of the public `MCPAdapter` surface reviewed here |
| Structured content | Via `ToolMessage` content | `ToolMessage.artifact["structured_content"]` (typed as `MCPToolArtifact`) |
| Error handling | `handle_tool_errors` boolean flag | `isError=True` → `status="error"` automatically; transport errors raise either way |
| Tool metadata | Ungrouped | Single `tool.metadata["mcp"]` namespace (`tool.annotations`, `_meta`, `server`) |
| Elicitation | Not built in | Automatic, via LangGraph `interrupt()` |
| Status | Mature, stable | **Beta** — "actively being worked on, so the API may change" (the module's own words) |

> 🎤 **Teaching Note (for you):** the honest takeaway for a client engagement — if you need
> resources, prompts, or interceptor-style middleware around MCP calls TODAY, `langchain-mcp-adapters`
> still does things `langchain.mcp` doesn't yet. If you're starting fresh and only need tools, the
> new built-in path is where LangChain is clearly headed, and it's what today's official docs lead
> with. Don't present this as a strict "old bad, new good" — it's "different surface area, and the
> new one isn't a full superset yet."